In [22]:
# RAG eith MongoDB - Data INgestion, retrieval and generation pipeline

# RAG Pipeline Overview (MongoDB Vector Search + Local Embeddings)
# 1) Ingest: Download PDF -> chunk text -> embed each chunk -> store {text, embedding, metadata} in MongoDB
# 2) Index: Create Atlas Vector Search index on embedding field
# 3) Retrieve: Embed user query -> $vectorSearch -> get top relevant chunks
# 4) (Optional) Generate: Send retrieved chunks as context to an LLM (Ollama) to produce final answer

In [11]:
# Install all required libraries for:
# - local embeddings (sentence-transformers)
# - PDF loading + chunking (langchain + pypdf)
# - MongoDB insert + vector search index (pymongo)
# - downloading the PDF (requests)
# Run once per environment.

In [1]:
import os
import time
import requests

from dotenv import load_dotenv
from pymongo import MongoClient
from pymongo.operations import SearchIndexModel

from sentence_transformers import SentenceTransformer
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()  # loads .env if present

print("Ready.")

/Users/himanshupaithane/Desktop/Projects/RAG-Mongo-VectorSearch/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Ready.


In [12]:
# Import dependencies and load environment variables (.env).
# .env will hold secrets like MONGODB_URI so we can commit this notebook safely without credentials.

In [2]:
# --- MongoDB ---

MONGODB_URI = os.getenv("MONGODB_URI") 

DB_NAME = "rag_db"
COLLECTION_NAME = "test"
INDEX_NAME = "vector_index"

# --- Embeddings model (local, free) ---
EMBED_MODEL_NAME = "BAAI/bge-small-en-v1.5"  # 384 dims
embedder = SentenceTransformer(EMBED_MODEL_NAME)
EMBED_DIM = embedder.get_sentence_embedding_dimension()

print("Embedding model:", EMBED_MODEL_NAME)
print("Embedding dim:", EMBED_DIM)

mongo_client = MongoClient(MONGODB_URI)
collection = mongo_client[DB_NAME][COLLECTION_NAME]

print("Connected to Mongo collection:", f"{DB_NAME}.{COLLECTION_NAME}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1767.99it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model: BAAI/bge-small-en-v1.5
Embedding dim: 384
Connected to Mongo collection: rag_db.test


In [13]:
# Define a reusable embedding function.
# Input: text chunk or user query
# Output: vector embedding (Python list of floats) stored in MongoDB and used for similarity search.

In [3]:
def get_embedding(text: str):
    text = text.strip()
    if not text:
        raise ValueError("text is empty/whitespace")
    # normalize_embeddings=True pairs nicely with cosine similarity
    return embedder.encode(text, normalize_embeddings=True).tolist()

# quick test
vec = get_embedding("RAG Technology")
print("Vector length:", len(vec))
print("First 5 values:", vec[:5])

Vector length: 384
First 5 values: [-0.05933227762579918, 0.031157083809375763, 0.04326363652944565, -0.07943200320005417, -0.00024953644606284797]


In [14]:
#  Download the PDF locally.
# PyPDFLoader is most reliable with a local file path (instead of directly loading from an HTTP URL).

In [4]:
pdf_url = "https://investors.mongodb.com/node/12236/pdf"
pdf_path = "mongodb_ai.pdf"

r = requests.get(pdf_url, timeout=60)
r.raise_for_status()
with open(pdf_path, "wb") as f:
    f.write(r.content)

print("Saved PDF to:", pdf_path, "bytes:", len(r.content))

Saved PDF to: mongodb_ai.pdf bytes: 139057


In [16]:
# Load the PDF and split it into smaller overlapping text chunks.
# Chunking is necessary because:
# - embeddings work best on manageable text sizes
# - retrieval returns the most relevant chunks instead of the whole document

In [5]:
loader = PyPDFLoader(pdf_path)
pages = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=20)
documents = text_splitter.split_documents(pages)

print("Pages loaded:", len(pages))
print("Chunks created:", len(documents))
print("Sample chunk:\n", documents[0].page_content[:500])

Pages loaded: 8
Chunks created: 88
Sample chunk:
 MongoDB, Inc. Announces First Quarter Fiscal 2025 Financial Results
May 30, 2024
First Quarter Fiscal 2025 Total Revenue of $450.6 million, up 22% Year-over-Year
Continued Strong Customer Growth with Over 49,200 Customers as of April 30, 2024
MongoDB Atlas Revenue up 32% Year-over-Year; 70% of Total Q1 Revenue


In [17]:
# Build the documents we will store in MongoDB.
# For each chunk we store:
# - text: the chunk content
# - embedding: vector representation of the chunk
# - metadata: source/page/chunk id for traceability

In [6]:
docs_to_insert = []
for i, doc in enumerate(documents):
    text = doc.page_content
    emb = get_embedding(text)
    docs_to_insert.append({
        "text": text,
        "embedding": emb,
        "metadata": {
            "source": pdf_url,
            "page": doc.metadata.get("page", None),
            "chunk": i
        }
    })

print("Docs prepared:", len(docs_to_insert))
print("Keys:", docs_to_insert[0].keys())
print("Embedding length:", len(docs_to_insert[0]["embedding"]))

Docs prepared: 88
Keys: dict_keys(['text', 'embedding', 'metadata'])
Embedding length: 384


In [18]:
# Insert all chunk documents into MongoDB.
# Optional: delete existing docs so reruns don’t create duplicates.
# After this step, MongoDB contains the knowledge base for retrieval.

In [7]:
# Optional reset for clean reruns:
collection.delete_many({})

result = collection.insert_many(docs_to_insert)
print("Inserted docs:", len(result.inserted_ids))

Inserted docs: 88


In [19]:
# Create an Atlas Vector Search index on the 'embedding' field.
# This index enables fast similarity search using $vectorSearch.
# Note: numDimensions must match EMBED_DIM exactly.

In [8]:
search_index_model = SearchIndexModel(
    definition={
        "fields": [{
            "type": "vector",
            "numDimensions": EMBED_DIM,
            "path": "embedding",
            "similarity": "cosine"
        }]
    },
    name=INDEX_NAME,
    type="vectorSearch"
)

# Create index (if it already exists, Atlas may throw an error; see note below)
try:
    collection.create_search_index(model=search_index_model)
    print("Index creation requested:", INDEX_NAME)
except Exception as e:
    print("Index create error (may already exist):", e)

print("Polling until index is queryable...")
while True:
    indices = list(collection.list_search_indexes(INDEX_NAME))
    if len(indices) and indices[0].get("queryable") is True:
        break
    time.sleep(5)

print(INDEX_NAME, "is ready.")

Index creation requested: vector_index
Polling until index is queryable...
vector_index is ready.


In [20]:
# Retrieval step (the "R" in RAG).
# - Embed the user query
# - Use MongoDB $vectorSearch to find the top-k most similar chunk embeddings
# - Return relevant text chunks (these become the context)

In [9]:
def vector_search(query: str, k: int = 5, num_candidates: int = 100):
    q_emb = get_embedding(query)
    pipeline = [
        {
            "$vectorSearch": {
                "index": INDEX_NAME,
                "path": "embedding",
                "queryVector": q_emb,
                "numCandidates": num_candidates,
                "limit": k
            }
        },
        {
            "$project": {
                "_id": 0,
                "text": 1,
                "metadata": 1,
                "score": {"$meta": "vectorSearchScore"}
            }
        }
    ]
    return list(collection.aggregate(pipeline))

results = vector_search("mongodb vector search", k=5)
print("Top results:", len(results))
print(results[0]["metadata"], "\n")
print(results[0]["text"][:600])

Top results: 5
{'source': 'https://investors.mongodb.com/node/12236/pdf', 'page': 0, 'chunk': 3} 

more of our customers. We also see a tremendous opportunity to win more legacy workloads, as AI has now become a catalyst to modernize these
applications. MongoDB's document-based architecture is particularly well-suited for the variety and scale of data required by AI-powered applications. 
We are confident MongoDB will be a substantial beneficiary of this next wave of application development."


In [21]:
# Build the context string from retrieved chunks (the "A" in RAG: Augmented context).
# This context_string is what we would pass to a language model to generate an answer.
# At this point we have a complete Retrieval + Context pipeline.
# The only missing part for full RAG is "G" (Generation) using an LLM like Ollama.

In [10]:
def build_context(query: str, k: int = 5):
    docs = vector_search(query, k=k)
    context = "\n\n".join([d["text"] for d in docs])
    return docs, context

query = "What are MongoDB's latest AI announcements?"
docs, context_string = build_context(query, k=5)

print("Query:", query)
print("\nContext preview:\n", context_string[:1200])

Query: What are MongoDB's latest AI announcements?

Context preview:
 MongoDB will host a conference call today, May 30, 2024, at 5:00 p.m. (Eastern Time) to discuss its financial results and business outlook. A live
webcast of the call will be available on the "Investor Relations" page of MongoDB's website at https://investors.mongodb.com. To access the call by

MongoDB continues to expand its AI ecosystem with the announcement of the MongoDB AI Applications Program (MAAP),

more of our customers. We also see a tremendous opportunity to win more legacy workloads, as AI has now become a catalyst to modernize these
applications. MongoDB's document-based architecture is particularly well-suited for the variety and scale of data required by AI-powered applications. 
We are confident MongoDB will be a substantial beneficiary of this next wave of application development."

millions of times since 2007, and there have been millions of builders trained through MongoDB University courses. To l